[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bigdata-com/bigdata-cookbook/blob/main/API_Tutorials/Search_API/Search_API_Phrase_Highlighting.ipynb)

# Example: Highlight Query-Relevant Phrases in Search Results

Client-facing demo on [Bigdata.com](https://bigdata.com): run a semantic Search API query, then **post-process** returned chunks to highlight phrases relevant to the query text.

**What this example shows**
- Run the same basic Search API pattern as `Search_API_Tutorial.ipynb` (section 1)
- Explain that the API returns relevant chunks and scores—not character-level highlight spans
- Compare three client-side highlighting approaches on the same results:
  1. Lexical-only (exact terms)
  2. LLM-only (semantic phrase spans via `gpt-5.4-nano`)
  3. Hybrid (exact + semantic, merged)

**Important:** Highlighting is optional post-processing. It does not change the search query, ranking, or API response.

In [1]:
from __future__ import annotations

import html
import json
import os
import re
from datetime import datetime, timezone
from time import perf_counter
from typing import Any

import requests
from dotenv import load_dotenv
from IPython.display import HTML, display
from print_helpers import print_search_results

## Setup & Configuration

Requires `BIGDATA_API_KEY` in `.env`.  
`OPENAI_API_KEY` is only needed for the LLM-only and hybrid sections.

In [2]:
load_dotenv()

API_BASE_URL = os.getenv("BIGDATA_API_BASE_URL", "https://api.bigdata.com")
API_KEY = os.getenv("BIGDATA_API_KEY")
if not API_KEY:
    raise ValueError("Set BIGDATA_API_KEY in .env")

SEARCH_ENDPOINT = f"{API_BASE_URL}/v1/search"

session = requests.Session()
session.headers.update({"Content-Type": "application/json", "X-API-KEY": API_KEY})
print("API key configured")

API key configured


## 1. Search API

Semantic search matches by meaning, not just keywords. A query like "cloud computing market growth" can match related phrasing.

Demo query: chunks in 2021 that match `the company expect revenue driven by cloud computing market growth`, mentioning **Apple** OR **Microsoft**.

In [3]:
# Tune these for your demo
START_DATE = "2021-01-01"
END_DATE = "2021-12-31"
TEXT = "the company expect revenue driven by cloud computing market growth"
ENTITY_ID = ["D8442A", "228D42"]  # Apple Inc, Microsoft
MAX_CHUNKS = 10
DISPLAY_TOP_N = 5  # how many chunks to highlight below

# Optional LLM settings (used in sections 3 and 4)
LLM_MODEL = "gpt-5.4-nano"

`auto_enrich_filters=False` keeps entity filtering under your control.  
`freshness_boost=0` avoids ranking bias toward newer documents.

In [4]:
search_query = {
    "query": {
        "text": TEXT,
        "auto_enrich_filters": False,
        "filters": {
            "timestamp": {
                "start": f"{START_DATE}T00:00:00Z",
                "end": f"{END_DATE}T23:59:59Z",
            },
            "entity": {
                "any_of": ENTITY_ID,
            },
        },
        "ranking_params": {
            "freshness_boost": 0
        },
        "max_chunks": MAX_CHUNKS,
    }
}

response = session.post(SEARCH_ENDPOINT, json=search_query)
data = response.json()
print_search_results(response, data)

✅ Status: 200
📄 Found 10 documents
💰 API Units Used: 1.0

--- Document 1 ---
Headline: Alphabet and Microsoft smash estimates with $110bn revenue haul
Source: Financial Times (Rank: RANK_1)
Date: 2021-10-26
Chunks: 1
First chunk preview: While still trailing Amazon's cloud offering, AWS, analysts see Microsoft's relationships with businesses, such as those using services such as Office...

--- Document 2 ---
Headline: Microsoft, Salesforce, Coupa Added to Goldman Sach's Conviction List, Positioned to Gain From Digital Transformation
Source: MT Newswires (Rank: RANK_1)
Date: 2021-03-09
Chunks: 1
First chunk preview: Goldman analysts see a path for double-digit topline growth for Microsoft in addition to continued margin expansion as the company's commercial cloud ...



### What the API returns (and what it does not)

Each result document includes chunks with `text`, `relevance`, and `sentiment`.

The Search API does **not** return character offsets or pre-highlighted HTML.  
Phrase highlighting below is entirely client-side post-processing of `chunk["text"]`.

In [5]:
def flatten_chunks(payload: dict[str, Any], limit: int | None = None) -> list[dict[str, Any]]:
    """Flatten documents into a list of chunk rows for highlighting."""
    rows: list[dict[str, Any]] = []
    for doc in payload.get("results", []):
        for chunk in doc.get("chunks", []):
            rows.append(
                {
                    "doc_id": doc.get("id"),
                    "headline": doc.get("headline", "N/A"),
                    "source": (doc.get("source") or {}).get("name", "N/A"),
                    "timestamp": (doc.get("timestamp") or "")[:10],
                    "cnum": chunk.get("cnum"),
                    "relevance": chunk.get("relevance"),
                    "sentiment": chunk.get("sentiment"),
                    "text": chunk.get("text") or "",
                }
            )
            if limit is not None and len(rows) >= limit:
                return rows
    return rows


STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "then", "else", "when", "at", "by",
    "for", "in", "of", "on", "to", "with", "from", "as", "is", "are", "was", "were",
    "be", "been", "being", "this", "that", "these", "those", "it", "its", "into",
    "about", "over", "under", "than", "so", "such", "via", "per",
}


def query_terms(query: str) -> list[str]:
    """Extract meaningful terms from the query for lexical highlighting."""
    tokens = re.findall(r"[A-Za-z0-9][A-Za-z0-9'\-]{1,}", query.lower())
    seen: set[str] = set()
    terms: list[str] = []
    for token in tokens:
        if token in STOPWORDS or token in seen:
            continue
        seen.add(token)
        terms.append(token)
    return terms


def find_span_occurrences(text: str, phrase: str) -> list[tuple[int, int]]:
    """Find case-insensitive, non-overlapping occurrences of phrase in text."""
    if not phrase or not text:
        return []
    spans: list[tuple[int, int]] = []
    pattern = re.compile(re.escape(phrase), flags=re.IGNORECASE)
    for match in pattern.finditer(text):
        spans.append((match.start(), match.end()))
    return spans


def validate_phrases_in_text(text: str, phrases: list[str]) -> list[str]:
    """Keep only phrases that appear verbatim (case-insensitive) in text."""
    valid: list[str] = []
    seen: set[str] = set()
    for phrase in phrases:
        cleaned = (phrase or "").strip()
        if not cleaned:
            continue
        key = cleaned.lower()
        if key in seen:
            continue
        if find_span_occurrences(text, cleaned):
            seen.add(key)
            valid.append(cleaned)
    return valid


def merge_spans(spans: list[tuple[int, int]]) -> list[tuple[int, int]]:
    """Merge overlapping/adjacent character spans."""
    if not spans:
        return []
    ordered = sorted(spans, key=lambda item: (item[0], item[1]))
    merged = [ordered[0]]
    for start, end in ordered[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))
    return merged


def spans_from_phrases(text: str, phrases: list[str]) -> list[tuple[int, int]]:
    spans: list[tuple[int, int]] = []
    for phrase in phrases:
        spans.extend(find_span_occurrences(text, phrase))
    return merge_spans(spans)


def highlight_text(text: str, spans: list[tuple[int, int]], mark_color: str = "#ffe566") -> str:
    """Render HTML with safe escaping and <mark> highlights."""
    if not text:
        return ""
    merged = merge_spans(spans)
    parts: list[str] = []
    cursor = 0
    for start, end in merged:
        if start < cursor:
            continue
        parts.append(html.escape(text[cursor:start]))
        parts.append(
            f'<mark style="background:{mark_color}; padding:0 2px;">'
            f"{html.escape(text[start:end])}</mark>"
        )
        cursor = end
    parts.append(html.escape(text[cursor:]))
    return "".join(parts)


def render_highlighted_chunks(
    rows: list[dict[str, Any]],
    query: str,
    phrase_fn,
    title: str,
    mark_color: str = "#ffe566",
) -> None:
    """Display document metadata + highlighted chunk text for a demo section."""
    blocks = [
        f"<h3>{html.escape(title)}</h3>",
        f"<p><b>Query:</b> {html.escape(query)}</p>",
    ]
    for idx, row in enumerate(rows, start=1):
        phrases = phrase_fn(query, row["text"])
        spans = spans_from_phrases(row["text"], phrases)
        highlighted = highlight_text(row["text"], spans, mark_color=mark_color)
        relevance = row.get("relevance")
        relevance_str = f"{relevance:.3f}" if isinstance(relevance, (int, float)) else "N/A"
        phrase_preview = ", ".join(phrases[:8]) if phrases else "(none)"
        blocks.append(
            "<div style='border:1px solid #ddd; border-radius:8px; "
            "padding:12px 14px; margin:10px 0;'>"
            f"<div><b>#{idx}</b> {html.escape(row['headline'])}</div>"
            f"<div style='color:#555; font-size:0.9em; margin:4px 0 8px;'>"
            f"{html.escape(row['source'])} | {html.escape(row['timestamp'])} | "
            f"relevance={html.escape(relevance_str)}</div>"
            f"<div style='margin-bottom:6px; font-size:0.9em;'>"
            f"<b>Matched phrases:</b> {html.escape(phrase_preview)}</div>"
            f"<div style='line-height:1.45;'>{highlighted}</div>"
            "</div>"
        )
    display(HTML("".join(blocks)))


chunks = flatten_chunks(data, limit=DISPLAY_TOP_N)
print(f"Prepared {len(chunks)} chunks for highlighting demos")
if chunks:
    print("First chunk preview:", chunks[0]["text"][:180], "...")

Prepared 5 chunks for highlighting demos
First chunk preview: While still trailing Amazon's cloud offering, AWS, analysts see Microsoft's relationships with businesses, such as those using services such as Office 365, as providing ample oppor ...


## 2. Approach A — Lexical-only highlighting

**How it works:** extract meaningful terms from the query and highlight exact matches in chunk text.

**Pros:** fast, no extra API cost, deterministic.  
**Cons:** misses paraphrases that semantic search still retrieves (e.g. "commercial cloud" vs "cloud computing").

In [6]:
def lexical_phrases(query: str, text: str) -> list[str]:
    """Return query terms that appear in the chunk text."""
    return validate_phrases_in_text(text, query_terms(query))


lexical_started_at = datetime.now(timezone.utc)
lexical_timer = perf_counter()
print(f"Started: {lexical_started_at.isoformat(timespec='seconds')}")

render_highlighted_chunks(
    rows=chunks,
    query=TEXT,
    phrase_fn=lexical_phrases,
    title="Lexical-only highlights",
    mark_color="#ffe566",
)

lexical_elapsed_seconds = perf_counter() - lexical_timer
lexical_ended_at = datetime.now(timezone.utc)
print(f"Ended:   {lexical_ended_at.isoformat(timespec='seconds')}")
print(f"Elapsed: {lexical_elapsed_seconds:.3f} seconds")

Started: 2026-07-23T20:23:38+00:00


Ended:   2026-07-23T20:23:38+00:00
Elapsed: 0.005 seconds


## 3. Approach B — LLM-only semantic highlighting

**How it works:** ask a small model (`gpt-5.4-nano`) to return short verbatim phrases from the chunk that best support the query intent. Every returned phrase is validated against the source text before highlighting.

**Pros:** catches semantic matches and paraphrases.  
**Cons:** needs `OPENAI_API_KEY`, small extra latency/cost.

Set `OPENAI_API_KEY` in `.env` before running this section.

In [7]:
from openai import OpenAI
from pydantic import BaseModel, Field


class HighlightPhrases(BaseModel):
    phrases: list[str] = Field(
        default_factory=list,
        description="Short verbatim phrases copied from the chunk that support the query",
    )


OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Set OPENAI_API_KEY in .env to run LLM highlighting")

openai_client = OpenAI(api_key=OPENAI_API_KEY)

LLM_SYSTEM = (
    "You extract short verbatim phrases from a document chunk that are relevant "
    "to a search query. Return only phrases that appear exactly in the chunk text "
    "(same wording; case may differ). Prefer 2-8 word phrases. Do not invent text."
)


def llm_phrases(query: str, text: str, max_phrases: int = 6) -> list[str]:
    """Ask the LLM for semantic highlight phrases, then validate against source text."""
    user_prompt = (
        f"Search query:\n{query}\n\n"
        f"Chunk text:\n{text}\n\n"
        f"Return up to {max_phrases} short verbatim phrases from the chunk "
        "that best match the query intent."
    )
    completion = openai_client.chat.completions.parse(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": LLM_SYSTEM},
            {"role": "user", "content": user_prompt},
        ],
        response_format=HighlightPhrases,
        temperature=0,
    )
    parsed = completion.choices[0].message.parsed
    raw_phrases = parsed.phrases if parsed else []
    return validate_phrases_in_text(text, raw_phrases)[:max_phrases]


llm_started_at = datetime.now(timezone.utc)
llm_timer = perf_counter()
print(f"Started: {llm_started_at.isoformat(timespec='seconds')}")

render_highlighted_chunks(
    rows=chunks,
    query=TEXT,
    phrase_fn=llm_phrases,
    title="LLM-only semantic highlights",
    mark_color="#9ad0ff",
)

llm_elapsed_seconds = perf_counter() - llm_timer
llm_ended_at = datetime.now(timezone.utc)
print(f"Ended:   {llm_ended_at.isoformat(timespec='seconds')}")
print(f"Elapsed: {llm_elapsed_seconds:.3f} seconds")

Started: 2026-07-23T20:23:38+00:00


Ended:   2026-07-23T20:23:44+00:00
Elapsed: 5.759 seconds


## 4. Approach C — Hybrid highlighting

**How it works:** union of lexical terms and LLM phrases, then merge overlapping spans.

**Pros:** keeps exact keyword hits and adds semantic paraphrases—usually the best client demo UX.  
**Cons:** still depends on OpenAI for the semantic half.

In [8]:
def hybrid_phrases(query: str, text: str) -> list[str]:
    """Combine lexical and LLM phrases; keep longer phrases first for display."""
    combined = lexical_phrases(query, text) + llm_phrases(query, text)
    # Deduplicate while preferring longer phrases in the preview list
    unique: dict[str, str] = {}
    for phrase in combined:
        key = phrase.lower()
        existing = unique.get(key)
        if existing is None or len(phrase) > len(existing):
            unique[key] = phrase
    return sorted(unique.values(), key=len, reverse=True)


hybrid_started_at = datetime.now(timezone.utc)
hybrid_timer = perf_counter()
print(f"Started: {hybrid_started_at.isoformat(timespec='seconds')}")

render_highlighted_chunks(
    rows=chunks,
    query=TEXT,
    phrase_fn=hybrid_phrases,
    title="Hybrid highlights (lexical + LLM)",
    mark_color="#b8f0c0",
)

hybrid_elapsed_seconds = perf_counter() - hybrid_timer
hybrid_ended_at = datetime.now(timezone.utc)
print(f"Ended:   {hybrid_ended_at.isoformat(timespec='seconds')}")
print(f"Elapsed: {hybrid_elapsed_seconds:.3f} seconds")

Started: 2026-07-23T20:23:44+00:00


Ended:   2026-07-23T20:23:49+00:00
Elapsed: 4.744 seconds


In [9]:
timing_results = (
    ("Lexical-only", lexical_started_at, lexical_ended_at, lexical_elapsed_seconds),
    ("LLM-only", llm_started_at, llm_ended_at, llm_elapsed_seconds),
    ("Hybrid", hybrid_started_at, hybrid_ended_at, hybrid_elapsed_seconds),
)

print("Approach timing summary (UTC)")
print("-" * 100)
for approach, started_at, ended_at, elapsed_seconds in timing_results:
    print(
        f"{approach:<14} "
        f"start={started_at.isoformat(timespec='seconds')}  "
        f"end={ended_at.isoformat(timespec='seconds')}  "
        f"elapsed={elapsed_seconds:.3f}s"
    )

Approach timing summary (UTC)
----------------------------------------------------------------------------------------------------
Lexical-only   start=2026-07-23T20:23:38+00:00  end=2026-07-23T20:23:38+00:00  elapsed=0.005s
LLM-only       start=2026-07-23T20:23:38+00:00  end=2026-07-23T20:23:44+00:00  elapsed=5.759s
Hybrid         start=2026-07-23T20:23:44+00:00  end=2026-07-23T20:23:49+00:00  elapsed=4.744s


## Quick adoption notes for clients

1. Call `/v1/search` as usual and keep `chunk["text"]`.
2. Choose a highlighter:
   - **Lexical** for zero extra dependencies
   - **LLM** when semantic paraphrases matter in the UI
   - **Hybrid** for the most complete demo experience
3. Always validate LLM phrases against the source text before rendering.
4. Always HTML-escape chunk text; wrap only validated spans in `<mark>`.

Docs: https://docs.bigdata.com/api-reference/search/search-documents